# 📓 Lab 00b — Guía de campo de los modelos clásicos

**Sesión:** [Sesión 0 — Fundamentos de Machine Learning](../sesiones/00-fundamentos-ml.md) · **Complemento del** [Lab 00](00_ml_clasico.ipynb)

El Lab 00 los puso a **competir**; esta guía los abre **uno por uno**: cada modelo con su intuición, su fórmula leída en palabras, su visualización característica y código comentado línea a línea. Está pensada para recorrerse en clase — cada sección se sostiene sola.

| # | Modelo | Su visualización característica |
|---|---|---|
| 1 | Regresión lineal | la recta y sus residuos |
| 2 | Regresión logística | la curva sigmoide sobre los datos |
| 3 | k-NN | el círculo de vecinos que votan |
| 4 | Árbol de decisión | el árbol de preguntas + sus cortes |
| 5 | Random forest | árboles individuales vs el voto |
| 6 | Gradient boosting | la predicción mejorando etapa a etapa |
| 7 | SVM | la calle más ancha y sus vectores de soporte |
| 8 | k-means | los centroides moviéndose hasta converger |
| 9 | PCA | las direcciones de máxima varianza |

In [ ]:
# Setup: numpy + matplotlib + scikit-learn (todo ya está en requirements.txt)
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # generador con semilla: reproducible

## 1. Regresión lineal — la recta que mejor pasa por los puntos

**Intuición.** Predecir un número trazando la recta que menos se equivoca. El "error" de cada punto es su **residuo**: la distancia vertical entre el valor real y la recta.

$$\hat{y} = w \cdot x + b$$

**Lectura:**

- ŷ = la predicción (un número)
- x = la feature de entrada
- w = la pendiente (cuánto sube el precio por cada metro extra)
- b = el intercepto (el valor base cuando x=0)
- **Entrenar** = encontrar la w y la b que minimizan el MSE — el error cuadrático medio, la misma loss que reencontrarás en la Sesión 1.

**Cuándo usarla:** relación aproximadamente lineal, pocas features, necesidad de interpretar ("cada metro² suma $2.5 millones"). **Cuándo falla:** relaciones curvas o con interacciones complejas.

In [ ]:
from sklearn.linear_model import LinearRegression

# ── Datos sintéticos: metros cuadrados → precio (millones) ─────────
metros = rng.uniform(40, 120, 60)                     # 60 apartamentos
precio = 2.5 * metros + 30 + rng.normal(0, 25, 60)    # relación real + ruido

# sklearn espera X como matriz (n_muestras, n_features): reshape(-1, 1)
reg = LinearRegression().fit(metros.reshape(-1, 1), precio)
w, b = reg.coef_[0], reg.intercept_
print(f'La recta aprendida: precio = {w:.2f}·metros + {b:.1f}')
print('(la relación real era 2.5·metros + 30 — la recuperó del ruido)')

# ── La visualización: la recta Y los residuos que minimiza ─────────
prediccion = reg.predict(metros.reshape(-1, 1))
xs = np.linspace(35, 125, 2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(metros, precio, s=30, zorder=3, label='datos')
ax.plot(xs, w * xs + b, color='crimson', lw=2.5, label='recta aprendida')
for m, real, pred in zip(metros, precio, prediccion):
    ax.plot([m, m], [real, pred], color='gray', lw=0.8, alpha=0.6)
ax.set_xlabel('metros cuadrados')
ax.set_ylabel('precio (millones)')
ax.set_title('Los segmentos grises son los RESIDUOS:\n'
             'la recta elegida es la que minimiza sus cuadrados')
ax.legend()
plt.show()

## 2. Regresión logística — la recta que responde con probabilidad

**Intuición.** Para clasificar (sí/no) no sirve predecir un número libre: queremos una **probabilidad**. La logística toma la misma combinación lineal de la regresión y la pasa por la **sigmoide**, que aplasta cualquier número al rango (0, 1).

$$p = \sigma(w \cdot x + b) = \frac{1}{1 + e^{-(wx+b)}}$$

**Lectura:**

- p = la probabilidad de la clase positiva
- σ = la sigmoide
- w, b = los mismos pesos y bias de la lineal
- **En palabras:** si wx+b es muy positivo → p ≈ 1; muy negativo → p ≈ 0; en la frontera (wx+b = 0) → p = 0.5.

> 🔗 **Este modelo ES la neurona de la [Sesión 1](../sesiones/01-fundamentos.md):** pesos + bias + sigmoide. Una red neuronal son muchas de estas apiladas con no-linealidades. Si entiendes esta sección, ya entiendes el 30% de la Sesión 1.

**Cuándo usarla:** clasificación con frontera ~lineal, baseline obligado, cuando importa la probabilidad calibrada. **Cuándo falla:** fronteras curvas (lo viste en moons).

In [ ]:
from sklearn.linear_model import LogisticRegression

# ── Datos sintéticos: horas de estudio → ¿aprueba el examen? ───────
horas = rng.uniform(0, 10, 90)
p_real = 1 / (1 + np.exp(-1.4 * (horas - 5)))     # la "regla" del mundo
aprueba = (rng.random(90) < p_real).astype(int)   # con azar: la vida es ruidosa

logit = LogisticRegression().fit(horas.reshape(-1, 1), aprueba)

# ── La visualización: los datos 0/1 y la sigmoide aprendida ────────
xs = np.linspace(0, 10, 300)
probs = logit.predict_proba(xs.reshape(-1, 1))[:, 1]   # P(aprobar) para cada x
frontera = xs[np.argmin(np.abs(probs - 0.5))]          # donde p cruza 0.5

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(horas, aprueba, s=30, alpha=0.6, label='estudiantes (0=pierde, 1=aprueba)')
ax.plot(xs, probs, color='crimson', lw=2.5, label='P(aprobar) aprendida')
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.axvline(frontera, color='gray', ls='--', lw=1.5,
           label=f'frontera de decisión: {frontera:.1f} horas')
ax.set_xlabel('horas de estudio')
ax.set_ylabel('probabilidad de aprobar')
ax.set_title('La sigmoide convierte la recta en probabilidad')
ax.legend()
plt.show()

## 3. k-NN — dime a qué te pareces y te diré quién eres

**Intuición.** k-NN no entrena nada: **memoriza** el dataset. Para clasificar un punto nuevo, busca sus **k vecinos más cercanos** y se queda con el voto mayoritario. Toda su "inteligencia" está en la distancia:

$$d(a, b) = \sqrt{\sum_i (a_i - b_i)^2}$$

*Lectura: la distancia euclidiana — la regla de Pitágoras generalizada: restar coordenada a coordenada, elevar al cuadrado, sumar y sacar raíz.*

**Cuándo usarlo:** pocos datos, fronteras raras, prototipos rápidos. **Cuándo falla:** muchos datos (buscar vecinos es caro), muchas features (en alta dimensión "todo queda lejos de todo"), features en escalas distintas (¡normalizar antes!).

> 🧩 **El hiperparámetro k controla la capacidad**: k=1 memoriza cada punto (frontera nerviosa → overfitting); k enorme promedia demasiado (frontera sosa → underfitting). Es el primer ejemplo del trade-off que la Sesión 1 llama generalización.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier

X, y = make_moons(n_samples=300, noise=0.25, random_state=42)

# ── Panel 1: cómo vota — el círculo de los k vecinos ───────────────
K = 7
consulta = np.array([[0.4, 0.35]])            # un punto nuevo, en zona disputada
knn = KNeighborsClassifier(n_neighbors=K).fit(X, y)
dist, idx = knn.kneighbors(consulta)          # distancias e índices de los K vecinos
votos = np.bincount(y[idx[0]], minlength=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu_r', s=15, alpha=0.5)
ax.scatter(X[idx[0], 0], X[idx[0], 1], facecolor='none', edgecolor='k',
           s=120, lw=1.5, label=f'los {K} vecinos')
ax.scatter(*consulta[0], marker='*', s=350, color='gold', edgecolor='k',
           zorder=5, label='punto nuevo')
circulo = plt.Circle(consulta[0], dist[0].max(), fill=False, ls='--', color='gray')
ax.add_patch(circulo)
ax.set_title(f'Votan {votos[0]} azules vs {votos[1]} rojos → clase {votos.argmax()}')
ax.legend(fontsize=8)

# ── Paneles 2-3: k controla la capacidad (frontera nerviosa vs sosa) ──
xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 200), np.linspace(-1.2, 1.8, 200))
malla = np.c_[xx.ravel(), yy.ravel()]
for ax, k in zip(axes[1:], (1, 25)):
    modelo = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    Z = modelo.predict(malla).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu_r')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu_r', s=10)
    ax.set_title(f'k={k}: frontera {"nerviosa (memoriza)" if k == 1 else "suave (promedia)"}')
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 4. Árbol de decisión — una cascada de preguntas si/no

**Intuición.** Un árbol juega "adivina quién": en cada nodo hace UNA pregunta sobre UNA feature (*¿largo del pétalo ≤ 2.45?*) y se ramifica. Elige cada pregunta buscando que los grupos resultantes queden lo más **puros** posible (cada uno dominado por una sola clase). La medida clásica de impureza es el índice **Gini**:

$$G = 1 - \sum_k p_k^2$$

*Lectura: p_k = proporción de la clase k en el nodo. Si el nodo es puro (todo una clase) → G = 0; si está 50/50 entre dos clases → G = 0.5 (con K clases el máximo es 1 − 1/K). Cada pregunta se elige para BAJAR G lo más posible.*

**Cuándo usarlo:** cuando la interpretabilidad manda — el árbol ES la explicación. **Cuándo falla:** solo, tiende a overfittear (por eso se limita `max_depth`) y sus cortes son siempre paralelos a los ejes — de ahí las fronteras "escalonadas" que viste.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Iris: el dataset clásico de 3 especies de flores. Usamos solo 2 features
# (largo y ancho del pétalo) para poder DIBUJAR las regiones.
iris = load_iris()
X_iris = iris.data[:, 2:]
nombres = ['largo pétalo (cm)', 'ancho pétalo (cm)']

# max_depth=2: solo 2 niveles de preguntas — pequeño a propósito, para leerlo
arbol = DecisionTreeClassifier(max_depth=2, random_state=0)
arbol.fit(X_iris, iris.target)
print(f'accuracy con solo 3 preguntas: {arbol.score(X_iris, iris.target):.2f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Panel 1: las regiones que crean sus cortes ─────────────────────
xx, yy = np.meshgrid(np.linspace(0.5, 7.5, 200), np.linspace(0, 3, 200))
Z = arbol.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax1.contourf(xx, yy, Z, alpha=0.25, cmap='viridis')
ax1.scatter(X_iris[:, 0], X_iris[:, 1], c=iris.target, cmap='viridis',
            s=20, edgecolors='k', linewidths=0.3)
ax1.set_xlabel(nombres[0]); ax1.set_ylabel(nombres[1])
ax1.set_title('Cada corte es paralelo a un eje:\npor eso las regiones son rectángulos')

# ── Panel 2: el árbol completo, legible — el modelo ES la explicación ──
plot_tree(arbol, ax=ax2, feature_names=nombres,
          class_names=list(iris.target_names), filled=True, fontsize=9)
ax2.set_title('El árbol de preguntas (gini = impureza del nodo)')
plt.tight_layout()
plt.show()

## 5. Random forest — la sabiduría de la multitud

**Intuición.** Un árbol solo es errático: cambia mucho si los datos cambian un poco. El random forest entrena **cientos de árboles**, cada uno con una muestra distinta de los datos (*bootstrap*: muestrear con reemplazo) y un subconjunto aleatorio de features — y los pone a **votar**:

$$\hat{y} = \mathrm{voto}\left(\mathrm{árbol}_1(x),\ \mathrm{árbol}_2(x),\ \dots,\ \mathrm{árbol}_B(x)\right)$$

*Lectura: B árboles deliberadamente distintos entre sí; el promedio de muchos modelos ruidosos pero diversos cancela sus errores individuales. La técnica se llama* bagging.

**Cuándo usarlo:** tabulares medianos, robusto casi sin tuning — un excelente "primer modelo serio". **Cuándo falla:** pierde la interpretabilidad del árbol individual; modelos grandes en memoria.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

bosque = RandomForestClassifier(n_estimators=100, random_state=0).fit(X, y)

# ── 3 árboles individuales vs el voto de los 100 ───────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, arbol_i in zip(axes[:3], bosque.estimators_[:3]):
    # cada estimador es un árbol entrenado con OTRA muestra bootstrap
    Z = arbol_i.predict(malla).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu_r')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu_r', s=8)
    ax.set_title('un árbol individual\n(errático a su manera)', fontsize=10)

Z_voto = bosque.predict_proba(malla)[:, 1].reshape(xx.shape)
axes[3].contourf(xx, yy, Z_voto, levels=16, alpha=0.5, cmap='RdBu_r')
axes[3].contour(xx, yy, Z_voto, levels=[0.5], colors='k', linewidths=1.5)
axes[3].scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu_r', s=8)
axes[3].set_title('el VOTO de los 100 árboles\n(suave y estable)', fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 6. Gradient boosting — árboles que corrigen los errores del anterior

**Intuición.** Donde el forest entrena árboles **en paralelo** e independientes, el boosting los entrena **en cadena**: cada árbol nuevo se especializa en los **residuos** (los errores) que dejaron los anteriores.

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

**Lectura:**

- F_m = el modelo tras m etapas
- h_m = un árbol pequeño entrenado sobre los errores de F_(m−1)
- η = el learning rate: cuánto confiar en cada corrección (lo reencontrarás en la Sesión 1).

**Cuándo usarlo:** el rey de los datos **tabulares** (sus implementaciones famosas: XGBoost, LightGBM, CatBoost). **Cuándo falla:** más sensible al tuning que el forest; secuencial = menos paralelizable.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# ── Regresión sobre una curva: ver la predicción MEJORAR por etapas ──
x_gb = np.sort(rng.uniform(0, 6, 120))
y_gb = np.sin(x_gb) + rng.normal(0, 0.25, 120)

gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.2,
                               max_depth=2, random_state=0)
gb.fit(x_gb.reshape(-1, 1), y_gb)

# staged_predict devuelve la predicción DESPUÉS de cada árbol agregado
xs = np.linspace(0, 6, 300).reshape(-1, 1)
etapas = list(gb.staged_predict(xs))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, m in zip(axes, (1, 5, 100)):
    ax.scatter(x_gb, y_gb, s=12, alpha=0.5)
    ax.plot(xs, etapas[m - 1], color='crimson', lw=2.5)
    ax.set_title(f'después de {m} árbol{"es" if m > 1 else ""}')
    ax.set_ylim(-1.8, 1.8)
fig.suptitle('Cada árbol nuevo corrige los residuos del modelo acumulado',
             fontweight='bold')
plt.tight_layout()
plt.show()

## 7. SVM — la calle más ancha entre las clases

**Intuición.** De todas las rectas que separan dos clases, la SVM elige la que deja la **calle más ancha** (el *margen*) entre ellas. Solo los puntos que tocan los bordes de la calle — los **vectores de soporte** — definen la frontera; el resto podría desaparecer sin cambiar nada.

$$\text{margen} = \frac{2}{\Vert w \Vert}$$

*Lectura: ‖w‖ es el tamaño del vector de pesos; maximizar el margen equivale a encontrar la w más "pequeña" que aún separa las clases. La intuición es lo importante: margen ancho = frontera con holgura = mejor generalización.*

Y su superpoder: el **truco del kernel** — medir similitudes como si los datos vivieran en un espacio de más dimensiones, sin calcularlo nunca. Así una SVM "lineal" dibuja fronteras curvas (la RBF que ganó en moons).

**Cuándo usarla:** datasets medianos, alta dimensión (texto clásico con TF-IDF). **Cuándo falla:** datasets grandes (entrenar escala mal) y necesita features escaladas.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# ── Dos clases separables: ver la calle y sus vectores de soporte ──
X_svm, y_svm = make_blobs(n_samples=80, centers=2, cluster_std=1.1,
                          random_state=6)
svm = SVC(kernel='linear', C=1.0).fit(X_svm, y_svm)

# decision_function = distancia (con signo) a la frontera:
#   0 = la frontera · ±1 = los bordes de la calle
gx, gy = np.meshgrid(
    np.linspace(X_svm[:, 0].min() - 1, X_svm[:, 0].max() + 1, 200),
    np.linspace(X_svm[:, 1].min() - 1, X_svm[:, 1].max() + 1, 200))
Z = svm.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(X_svm[:, 0], X_svm[:, 1], c=y_svm, cmap='RdBu_r', s=30)
ax.contour(gx, gy, Z, levels=[-1, 0, 1], colors='k',
           linestyles=['--', '-', '--'], linewidths=[1, 2, 1])
ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
           s=220, facecolor='none', edgecolor='gold', linewidths=2.5,
           label=f'vectores de soporte ({len(svm.support_vectors_)})')
ax.set_title('La frontera (línea sólida) y la calle (punteadas):\n'
             'solo los puntos dorados la definen')
ax.legend()
plt.show()

## 8. k-means — mover centroides hasta que dejen de moverse

**Intuición** (no supervisado: aquí NO hay etiquetas). Elige k "imanes" (centroides) al azar y repite dos pasos hasta converger: (1) cada punto se une a su imán más cercano; (2) cada imán se muda al centro de sus puntos. El objetivo que minimiza:

$$J = \sum_i \min_k \Vert x_i - \mu_k \Vert^2$$

**Lectura:**

- μ_k = el centroide del grupo k
- J = la suma, para cada punto, del cuadrado de su distancia al centroide más cercano
- **En palabras:** "buenos grupos" = puntos cerca de su centroide.

Aquí lo implementamos **a mano en numpy** para ver los centroides moverse — son ~10 líneas.

In [ ]:
from sklearn.datasets import make_blobs

X_km, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.0,
                     random_state=4)

K = 3
# Inicialización: k centroides elegidos al azar ENTRE los puntos
centroides = X_km[rng.choice(len(X_km), K, replace=False)].copy()
historia = [centroides.copy()]

for iteracion in range(6):
    # Paso 1 — asignar: distancia de cada punto a cada centroide → el más cercano
    distancias = np.linalg.norm(X_km[:, None, :] - centroides[None, :, :], axis=2)
    asignacion = distancias.argmin(axis=1)          # (300,): grupo de cada punto
    # Paso 2 — mover: cada centroide al promedio de sus puntos
    nuevos = np.array([X_km[asignacion == k].mean(axis=0) for k in range(K)])
    if np.allclose(nuevos, centroides):             # ¿ya nadie se movió?
        break
    centroides = nuevos
    historia.append(centroides.copy())

print(f'convergió en {len(historia)} iteraciones')

# ── Ver la convergencia: inicio → iteraciones → final ──────────────
fig, axes = plt.subplots(1, len(historia), figsize=(4 * len(historia), 3.8))
for i, (ax, centros) in enumerate(zip(np.atleast_1d(axes), historia)):
    d = np.linalg.norm(X_km[:, None, :] - centros[None, :, :], axis=2)
    grupos = d.argmin(axis=1)
    ax.scatter(X_km[:, 0], X_km[:, 1], c=grupos, cmap='viridis', s=10)
    ax.scatter(*centros.T, marker='X', s=250, c='red', edgecolors='k', zorder=5)
    ax.set_title('inicio (al azar)' if i == 0 else f'iteración {i}')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 9. PCA — encontrar los ejes que importan

**Intuición** (no supervisado). Los datos rara vez "usan" todas sus dimensiones: en una nube alargada, casi toda la información vive a lo largo de UNA dirección. PCA encuentra esas direcciones — las **componentes principales** — ordenadas por cuánta **varianza** (dispersión) capturan, y proyecta los datos sobre las primeras.

*No hay fórmula nueva que memorizar: la componente 1 es la dirección donde los datos más varían; la 2 es la perpendicular que más varía de lo que queda; y así. El número a reportar es la varianza explicada: "con 1 de las 2 dimensiones conservo el 95% de la información".*

**Para qué:** visualizar datos de muchas dimensiones (como los dígitos del Lab 00), comprimir features antes de otro modelo, eliminar redundancia. **Su límite:** solo captura direcciones RECTAS — los embeddings de la Sesión 3 son su generalización aprendida.

In [ ]:
from sklearn.decomposition import PCA

# ── Una nube correlacionada: dos features que casi cuentan lo mismo ──
X_pca = rng.multivariate_normal([0, 0], [[3.0, 2.2], [2.2, 2.0]], 400)

pca = PCA(n_components=2).fit(X_pca)
print('varianza explicada por componente:', pca.explained_variance_ratio_.round(3))

# Proyectar sobre SOLO la componente 1 y volver al espacio original
# (para dibujar dónde caería cada punto)
proyectado = pca.transform(X_pca)[:, :1] @ pca.components_[:1] + pca.mean_

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.scatter(X_pca[:, 0], X_pca[:, 1], s=10, alpha=0.4)
for comp, var, color in zip(pca.components_, pca.explained_variance_, ['crimson', 'seagreen']):
    # cada flecha: la dirección de la componente, escalada por su varianza
    ax1.annotate('', xy=pca.mean_ + comp * np.sqrt(var) * 2.2, xytext=pca.mean_,
                 arrowprops=dict(arrowstyle='->', color=color, lw=3))
ax1.set_title('Las 2 componentes principales:\nroja = donde los datos MÁS varían')
ax1.set_aspect('equal')

ax2.scatter(X_pca[:, 0], X_pca[:, 1], s=10, alpha=0.15, label='original (2D)')
ax2.scatter(proyectado[:, 0], proyectado[:, 1], s=10, color='crimson',
            label='proyectado sobre la componente 1')
ax2.set_title(f'Comprimido a 1 dimensión conservando el '
              f'{pca.explained_variance_ratio_[0]:.0%} de la varianza')
ax2.set_aspect('equal')
ax2.legend()
plt.tight_layout()
plt.show()

## 🎯 Cierre — el mapa mental para la clase

| Modelo | En una frase | Su hiperparámetro clave |
|---|---|---|
| Regresión lineal | la recta que minimiza los residuos² | — |
| Regresión logística | la recta + sigmoide = probabilidad (la neurona) | regularización C |
| k-NN | el voto de los k vecinos más cercanos | k (capacidad) |
| Árbol | cascada de preguntas que purifican (Gini↓) | max_depth |
| Random forest | cientos de árboles diversos que votan | n_estimators |
| Gradient boosting | árboles en cadena corrigiendo residuos | learning_rate + n_estimators |
| SVM | la calle más ancha; kernels para curvarla | C y el kernel |
| k-means | asignar y mover centroides hasta converger | k |
| PCA | proyectar sobre las direcciones de máxima varianza | n_components |

**Reto:** vuelve al [Lab 00](00_ml_clasico.ipynb) y, con lo aprendido aquí, explica *por qué* cada modelo dibujó su frontera con esa forma en moons. Una frase por modelo.

**Siguiente parada:** la [Sesión 1](../sesiones/01-fundamentos.md) — la regresión logística de la sección 2 se convierte en neurona, y apilar neuronas resuelve lo que ninguna recta pudo.